# PocketVoice — Live-talk Miku

**На телефоне:** ☰ слева сверху → **Runtime** → **Change runtime type** → **T4 GPU** → Save. Потом тапни ▶ слева от кода ниже. Жди 10-15 мин.

In [ ]:
import os, sys, subprocess, time, threading, re, urllib.request, json

os.chdir('/content/')
if not os.path.exists('/content/voice-changer'):
    print('[1/4] clone w-okada...')
    subprocess.run(['git','clone','-q','https://github.com/w-okada/voice-changer.git'], check=True)
os.chdir('/content/voice-changer/server')
subprocess.run(['apt-get','-y','install','libportaudio2','-qq'], check=True)

print('[2/4] pip deps (~5-7 min)...')
subprocess.run([sys.executable,'-m','pip','install','-q','faiss-gpu','fairseq','pyworld','--no-build-isolation'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'], check=True)

print('[3/4] Miku model + cloudflared...')
slot_dir = '/content/voice-changer/server/model_dir/RVC/0'
os.makedirs(slot_dir, exist_ok=True)
miku_pth = f'{slot_dir}/miku.pth'
miku_idx = f'{slot_dir}/miku.index'
if not os.path.exists(miku_pth):
    urllib.request.urlretrieve('https://huggingface.co/NoCrypt/miku_RVC/resolve/main/1a_miku_default_rvc_(aple)/miku_default_rvc.pth', miku_pth)
if not os.path.exists(miku_idx):
    urllib.request.urlretrieve('https://huggingface.co/NoCrypt/miku_RVC/resolve/main/1a_miku_default_rvc_(aple)/added_IVF4457_Flat_nprobe_1_miku_default_rvc_v2.index', miku_idx)
params = {'slotIndex':0,'voiceChangerType':'RVC','name':'Miku','description':'Hatsune Miku RVC v2','modelFile':'miku.pth','indexFile':'miku.index','defaultTune':12,'defaultIndexRatio':0.75,'defaultProtect':0.33,'sampleRate':40000,'modelType':'pyTorchRVCv2','embChannels':768,'embOutputLayer':12,'useFinalProj':False,'f0':True}
with open(f'{slot_dir}/params.json','w') as f: json.dump(params, f)
if not os.path.exists('/content/cloudflared'):
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64','/content/cloudflared')
    os.chmod('/content/cloudflared', 0o755)

print('[4/4] start server + tunnel...')
server = subprocess.Popen([sys.executable,'MMVCServerSIO.py','-p','18888','--https','False','--content_vec_500','pretrain/checkpoint_best_legacy_500.pt','--hubert_base','hubert_base.pt','--rmvpe','rmvpe.pt','--colab','True'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in iter(server.stdout.readline,''):
    print(line, end='')
    if 'Uvicorn running' in line or '18888' in line: time.sleep(3); break
    if server.poll() is not None: print('[!] server died'); raise SystemExit(1)

tunnel = subprocess.Popen(['/content/cloudflared','tunnel','--url','http://localhost:18888','--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
url_re = re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com')
public_url = None
for line in iter(tunnel.stdout.readline,''):
    print(line, end='')
    m = url_re.search(line)
    if m: public_url = m.group(0); break

if public_url:
    print('\n' + '='*70)
    print(f'  >>> ОТКРОЙ ЭТУ ССЫЛКУ В НОВОЙ ВКЛАДКЕ CHROME: <<<')
    print(f'  {public_url}')
    print('='*70)
    print('  Slot 0 = Miku → жми Start → разреши mic → говори\n')

def tail(p, tag):
    for ln in iter(p.stdout.readline,''): print(f'[{tag}] {ln}', end='')
threading.Thread(target=tail, args=(server,'srv'), daemon=True).start()
threading.Thread(target=tail, args=(tunnel,'tun'), daemon=True).start()

print('[ready] держу. НЕ ЗАКРЫВАЙ ВКЛАДКУ COLAB.')
while True:
    time.sleep(60)
    if server.poll() is not None: print('[!] server died'); break
    if tunnel.poll() is not None: print('[!] tunnel died'); break